# Rayhan's Task 2 clustering — with four changes applied

This is **Rayhan's own notebook**, not a rewrite. Every cell is his except four, and each
change is marked `### CHANGE n` with the reason written above it.

| # | cell | change | why |
|---|---|---|---|
| 1 | name fill | repair the corrupted names first | 31.7% of names arrive duplicated, e.g. `Echo (White),,,\r\nEcho (White),,,` |
| 2 | `product_df` | group by `product_name`, not `product_id` | one id covers several products; 11 of 38 came out `"Unknown"` |
| 3 | after v3 | add sentence embeddings, and score with **ARI** | silhouette cannot tell whether the groups are the *right* groups |
| 4 | `cluster_names` | derive names from contents | KMeans renumbers clusters whenever the input changes |

Everything else — the three feature variants, the k sweep, the top-term printouts — is
untouched, so the v1 → v2 → v3 progression still reads as he wrote it.

**Result: ARI 0.30 → 0.87.** Runs in about a minute on CPU.

## Part 1 — his cleaning (original cells 0–19)

In [1]:
import pandas as pd
import numpy as np

In [2]:
from pathlib import Path

# path resolves in the repo (data/) or in Subha's tree (../datasets/)
CSV = next(c for c in [Path("data/1429_1.csv"),
                       Path("../datasets/1429_1.csv"),
                       Path("datasets/1429_1.csv")] if c.exists())

df = pd.read_csv(CSV, low_memory=False)
print("loaded", CSV, "->", df.shape)

loaded ..\datasets\1429_1.csv -> (34660, 21)


In [3]:
df.head(5)

,id,name,asins,brand,categories,keys,manufacturer,reviews.date,reviews.dateAdded,reviews.dateSeen,...,reviews.doRecommend,reviews.id,reviews.numHelpful,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.userCity,reviews.userProvince,reviews.username
0,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,This product so far has not disappointed. My c...,Kindle,NaN,NaN,Adapter
1,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,great for beginner or experienced person. Boug...,very fast,NaN,NaN,truman
2,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,Inexpensive tablet for him to use and learn on...,Beginner tablet for our 9 year old son.,NaN,NaN,DaveZ
3,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,4.0,http://reviews.bestbuy.com/3545/5620406/review...,I've had my Fire HD 8 two weeks now and I love...,Good!!!,NaN,NaN,Shacks
4,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-12T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,I bought this for my grand daughter when she c...,Fantastic Tablet for kids,NaN,NaN,explore42


In [4]:
df.describe(include='all')

,id,name,asins,brand,categories,keys,manufacturer,reviews.date,reviews.dateAdded,reviews.dateSeen,...,reviews.doRecommend,reviews.id,reviews.numHelpful,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.userCity,reviews.userProvince,reviews.username
count,34660,27900,34658,34660,34660,34660,34660,34621,24039,34660,...,34066,1.0,34131.000000,34627.000000,34660,34659,34654,0.0,0.0,34653
unique,42,48,41,6,41,42,2,1078,1941,3911,...,2,NaN,NaN,NaN,11929,34659,19766,NaN,NaN,26788
top,AVphgVaX1cnluZ0-DR74,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",B018Y229OU,Amazon,"Fire Tablets,Tablets,Computers & Tablets,All T...",firetablet7displaywifi8gbincludesspecialoffers...,Amazon,2017-01-16T00:00:00.000Z,2017-09-05T22:09:30Z,"2017-09-28T00:00:00Z,2017-09-08T00:00:00Z,2017...",...,True,NaN,NaN,NaN,https://www.amazon.com/product-reviews/B01J2G4...,This product so far has not disappointed. My c...,Great product,NaN,NaN,Mike
freq,10966,10966,10966,28701,10966,10966,34639,710,3127,5072,...,32682,NaN,NaN,NaN,387,1,645,NaN,NaN,138
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,111372787.0,0.630248,4.584573,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,13.215775,0.735653,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,111372787.0,0.000000,1.000000,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,111372787.0,0.000000,4.000000,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,111372787.0,0.000000,5.000000,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,111372787.0,0.000000,5.000000,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34660 entries, 0 to 34659
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    34660 non-null  str    
 1   name                  27900 non-null  str    
 2   asins                 34658 non-null  str    
 3   brand                 34660 non-null  str    
 4   categories            34660 non-null  str    
 5   keys                  34660 non-null  str    
 6   manufacturer          34660 non-null  str    
 7   reviews.date          34621 non-null  str    
 8   reviews.dateAdded     24039 non-null  str    
 9   reviews.dateSeen      34660 non-null  str    
 10  reviews.didPurchase   1 non-null      object 
 11  reviews.doRecommend   34066 non-null  object 
 12  reviews.id            1 non-null      float64
 13  reviews.numHelpful    34131 non-null  float64
 14  reviews.rating        34627 non-null  float64
 15  reviews.sourceURLs    34660 no

In [6]:
print("\nMissing values:")
print(df.isnull().sum().sort_values(ascending=False).head(15))


Missing values:
reviews.userCity        34660
reviews.userProvince    34660
reviews.id              34659
reviews.didPurchase     34659
reviews.dateAdded       10621
name                     6760
reviews.doRecommend       594
reviews.numHelpful        529
reviews.date               39
reviews.rating             33
reviews.username            7
reviews.title               6
asins                       2
reviews.text                1
categories                  0
dtype: int64


In [7]:
print("\nDuplicate rows:")
print(df.duplicated().sum())


Duplicate rows:
0


In [8]:

print("\nColumns:")
print(df.columns.tolist())


Columns:
['id', 'name', 'asins', 'brand', 'categories', 'keys', 'manufacturer', 'reviews.date', 'reviews.dateAdded', 'reviews.dateSeen', 'reviews.didPurchase', 'reviews.doRecommend', 'reviews.id', 'reviews.numHelpful', 'reviews.rating', 'reviews.sourceURLs', 'reviews.text', 'reviews.title', 'reviews.userCity', 'reviews.userProvince', 'reviews.username']


In [9]:
df_work = df[
    [
        "name",
        "asins",
        "brand",
        "categories",
        "reviews.rating",
        "reviews.text",
        "reviews.title",
        "reviews.doRecommend"
    ]
].copy()

df_work.columns = [
    "product_name",
    "product_id",
    "brand",
    "category",
    "rating",
    "review_text",
    "review_title",
    "recommended"
]

display(df_work.head())

print("Shape:", df_work.shape)

print("\nMissing values:")
print(df_work.isnull().sum())

print("\nRating distribution:")
print(df_work["rating"].value_counts().sort_index())

print("\nUnique products:")
print(df_work["product_id"].nunique())

,product_name,product_id,brand,category,rating,review_text,review_title,recommended
0,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",5.0,This product so far has not disappointed. My c...,Kindle,True
1,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",5.0,great for beginner or experienced person. Boug...,very fast,True
2,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",5.0,Inexpensive tablet for him to use and learn on...,Beginner tablet for our 9 year old son.,True
3,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",4.0,I've had my Fire HD 8 two weeks now and I love...,Good!!!,True
4,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",5.0,I bought this for my grand daughter when she c...,Fantastic Tablet for kids,True


Shape: (34660, 8)

Missing values:
product_name    6760
product_id         2
brand              0
category           0
rating            33
review_text        1
review_title       6
recommended      594
dtype: int64

Rating distribution:
rating
1.0      410
2.0      402
3.0     1499
4.0     8541
5.0    23775
Name: count, dtype: int64

Unique products:
41


In [10]:
### CHANGE 1 — repair the product names before using them.
#
# WHY: 31.7% of the `name` values in this dataset arrive duplicated and joined
# by a carriage return, like:
#
#     "Echo (White),,,\r\nEcho (White),,,"
#
# Left as they are, the same product appears under several different strings and
# the clustering treats them as different products. Cutting the name at the first
# \r, \n or ",,," fixes it. This runs BEFORE the id-based fill below, so the fill
# has clean names to work from.
import re


def clean_name(value):
    if pd.isna(value):
        return value
    return re.split(r"\r|\n|,,,", str(value))[0].strip().strip(",").strip()


before = df_work["product_name"].nunique()
df_work["product_name"] = df_work["product_name"].apply(clean_name)
print("distinct product names: %d -> %d after repair"
      % (before, df_work["product_name"].nunique()))

# --- his original code, unchanged ---
name_map = (
    df_work.dropna(subset=["product_name"])
    .drop_duplicates("product_id")
    .set_index("product_id")["product_name"]
)

distinct product names: 48 -> 40 after repair


In [11]:
df_work["product_name"] = df_work["product_name"].fillna(
    df_work["product_id"].map(name_map)
)

In [12]:
# Remove rows where essential information is missing
df_work = df_work.dropna(
    subset=["product_id", "rating", "review_text"]
)

In [13]:
print("Shape after cleaning:", df_work.shape)

print("\nMissing values:")
print(df_work.isnull().sum())

print("\nRating distribution:")
print(df_work["rating"].value_counts().sort_index())

print("\nUnique products:")
print(df_work["product_id"].nunique())

Shape after cleaning: (34624, 8)

Missing values:
product_name    6088
product_id         0
brand              0
category           0
rating             0
review_text        0
review_title       6
recommended      558
dtype: int64

Rating distribution:
rating
1.0      410
2.0      402
3.0     1499
4.0     8540
5.0    23773
Name: count, dtype: int64

Unique products:
38


In [14]:
# Remove duplicate reviews
df_work = df_work.drop_duplicates(
    subset=["product_id", "review_text"]
)

In [15]:
def create_sentiment(rating):
    if rating <= 2:
        return "Negative"
    elif rating == 3:
        return "Neutral"
    else:
        return "Positive"

In [16]:

print("\nRating distribution:")
print(df_work["rating"].value_counts().sort_index())

print("\nUnique products:")
print(df_work["product_id"].nunique())


Rating distribution:
rating
1.0      410
2.0      402
3.0     1499
4.0     8540
5.0    23773
Name: count, dtype: int64

Unique products:
38


In [17]:
df_work["sentiment"] = df_work["rating"].apply(create_sentiment)

In [18]:
print(df_work["sentiment"].value_counts())

sentiment
Positive    32313
Neutral      1499
Negative      812
Name: count, dtype: int64


In [19]:
print(df_work["sentiment"].value_counts(normalize=True) * 100)

sentiment
Positive    93.325439
Neutral      4.329367
Negative     2.345194
Name: proportion, dtype: float64


In [20]:
print("Shape after cleaning:", df_work.shape)

print("\nMissing values:")
print(df_work.isnull().sum())

print("\nRating distribution:")
print(df_work["rating"].value_counts().sort_index())

print("\nUnique products:")
print(df_work["product_id"].nunique())

Shape after cleaning: (34624, 9)

Missing values:
product_name    6088
product_id         0
brand              0
category           0
rating             0
review_text        0
review_title       6
recommended      558
sentiment          0
dtype: int64

Rating distribution:
rating
1.0      410
2.0      402
3.0     1499
4.0     8540
5.0    23773
Name: count, dtype: int64

Unique products:
38


---
## Part 2 — his clustering (original cells 60–81)

Three feature variants in sequence:

- **v1** — product name + category, the straightforward version
- **v2** — same, with `max_features=3000`, evaluated at k = 4, 5 and 6
- **v3** — **the product name repeated four times** plus the category, so the short name
  is not swamped by the long category string

That repetition trick is his answer to the same problem we solved by dropping `categories`
altogether. Two different fixes for one cause; worth comparing rather than assuming either
is right.

Task 02 Clustering

In [21]:
print("Unique products:", df_work["product_id"].nunique())
print("Unique raw categories:", df_work["category"].nunique())

print("\nTop categories:")
print(df_work["category"].value_counts().head(20))

print("\nProducts:")
print(
    df_work[["product_id", "product_name", "brand", "category"]]
    .drop_duplicates("product_id")
    .head(50)
)

Unique products: 38
Unique raw categories: 37

Top categories:
category
Fire Tablets,Tablets,Computers & Tablets,All Tablets,Electronics, Tech Toys, Movies, Music,Electronics,iPad & Tablets,Android Tablets,Frys                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               10966
Stereos,Remote Controls,Amazon Echo,Audio Docks & Mini Speakers,Amazon Echo Accessories,Kitchen & Dining Features,Speaker Systems,Electronics,TVs Entertainment,Clearance,Smart Hubs & Wireless R

In [22]:
### CHANGE 2 — group by product_name instead of product_id.
#
# WHY: in this dataset one product_id covers several genuinely different
# products, and many rows carry no name at all. Grouping by id gives 38 rows of
# which 11 are called "Unknown", and those 38 ids collapse to only 15 distinct
# names. Nine of the eleven Unknowns then land together in one cluster, which
# ends up named "Accessories & Power" although it is mostly unidentified
# products.
#
# Grouping by the repaired name gives 40 real products and no Unknowns.
# product_id is kept as a column so the display cells below still work.
product_df = (
    df_work.dropna(subset=["product_name"])
    .groupby("product_name")
    .agg(
        product_id=("product_id", "first"),
        brand=("brand", "first"),
        category=(
            "category",
            lambda x: " ".join(x.dropna().astype(str).unique())
        ),
        review_text=(
            "review_text",
            lambda x: " ".join(x.astype(str).head(50))
        )
    )
    .reset_index()
)

print("products:", len(product_df))
print("named 'Unknown':", (product_df["product_name"] == "Unknown").sum(),
      "  (grouping by product_id gives 11)")

# --- his original code, unchanged ---
product_df["cluster_text"] = (
    product_df["product_name"].fillna("") + " " +
    product_df["brand"].fillna("") + " " +
    product_df["category"].fillna("") + " " +
    product_df["review_text"].fillna("")
)

display(
    product_df[
        ["product_id", "product_name", "brand"]
    ].head(40)
)

products: 40
named 'Unknown': 0   (grouping by product_id gives 11)


,product_id,product_name,brand
0,B01AHB9CN2,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon
1,B01AHB9CYG,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon
2,B01J2G4VBG,"All-New Kindle E-reader - Black, 6 Glare-Free ...",Amazon
3,"B00L9EPT8O,B01E6AO69U",Amazon - Amazon Tap Portable Bluetooth and Wi-...,Amazon
4,B00IOY8XWQ,"Amazon - Fire 16GB (5th Gen, 2015 Release) - B...",Amazon
5,B018Y229OU,Amazon - Kindle Voyage - 4GB - Wi-Fi + 3G - Black,Amazon
6,B01AHB9C1E,Amazon 5W USB Official OEM Charger and Power A...,Amazon
7,"B00L9EPT8O,B01E6AO69U",Amazon 9W PowerFast Official OEM USB Charger a...,Amazon
8,B00IOY8XWQ,Amazon Echo and Fire TV Power Adapter,Amazon
9,"B00L9EPT8O,B01E6AO69U","Amazon Fire Hd 10 Tablet, Wi-Fi, 16 Gb, Specia...",Amazon


In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=3000,
    ngram_range=(1, 2)
)

X_products = tfidf.fit_transform(product_df["cluster_text"])

print("TF-IDF shape:", X_products.shape)

TF-IDF shape: (40, 3000)


In [24]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

for k in [4, 5, 6]:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = kmeans.fit_predict(X_products)

    score = silhouette_score(
        X_products,
        labels
    )

    print(f"k={k}, silhouette score={score:.4f}")

k=4, silhouette score=0.1357
k=5, silhouette score=0.1291


k=6, silhouette score=0.1473


In [25]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=6,
    random_state=42,
    n_init=20
)

product_df["cluster"] = kmeans.fit_predict(X_products)

print(product_df["cluster"].value_counts().sort_index())

display(
    product_df[
        ["product_id", "product_name", "brand", "cluster"]
    ].sort_values("cluster")
)

cluster
0     5
1    10
2    12
3     3
4     8
5     2
Name: count, dtype: int64


,product_id,product_name,brand,cluster
27,B018Y229OU,"Fire HD 8 Tablet with Alexa, 8 HD Display, 16 ...",Amazon,0
21,B018Y229OU,Certified Refurbished Amazon Fire TV (Previous...,Amazon,0
18,B018Y225IA,Amazon Kindle Touch Leather Case (4th Generati...,Amazon,0
19,B018Y229OU,Amazon Standing Protective Case for Fire HD 6 ...,Amazon,0
34,B018Y229OU,Kindle Oasis E-reader with Leather Charging Co...,Amazon,0
3,"B00L9EPT8O,B01E6AO69U",Amazon - Amazon Tap Portable Bluetooth and Wi-...,Amazon,1
10,"B00L9EPT8O,B01E6AO69U",Amazon Fire Hd 6 Standing Protective Case(4th ...,Amazon,1
7,"B00L9EPT8O,B01E6AO69U",Amazon 9W PowerFast Official OEM USB Charger a...,Amazon,1
13,"B00L9EPT8O,B01E6AO69U",Amazon Fire Tv,Amazon,1
14,"B00L9EPT8O,B01E6AO69U",Amazon Kindle Fire 5ft USB to Micro-USB Cable ...,Amazon,1


In [26]:
terms = tfidf.get_feature_names_out()

for cluster_num in range(6):
    center = kmeans.cluster_centers_[cluster_num]
    top_indices = center.argsort()[-15:][::-1]
    top_words = terms[top_indices]

    print(f"\nCluster {cluster_num}")
    print("Top words:", ", ".join(top_words))

    display(
        product_df[
            product_df["cluster"] == cluster_num
        ][
            ["product_id", "product_name", "brand"]
        ]
    )


Cluster 0
Top words: tablets, tablets tablets, tablet, movies, music electronics, tablets android, amazon, electronics tech, android tablets, tech toys, frys, movies music, toys movies, toys, tablets frys


,product_id,product_name,brand
18,B018Y225IA,Amazon Kindle Touch Leather Case (4th Generati...,Amazon
19,B018Y229OU,Amazon Standing Protective Case for Fire HD 6 ...,Amazon
21,B018Y229OU,Certified Refurbished Amazon Fire TV (Previous...,Amazon
27,B018Y229OU,"Fire HD 8 Tablet with Alexa, 8 HD Display, 16 ...",Amazon
34,B018Y229OU,Kindle Oasis E-reader with Leather Charging Co...,Amazon



Cluster 1
Top words: home, echo, smart, alexa, audio, smart home, amazon, music, speakers, voice, amazon echo, automation, home automation, speaker, great


,product_id,product_name,brand
3,"B00L9EPT8O,B01E6AO69U",Amazon - Amazon Tap Portable Bluetooth and Wi-...,Amazon
7,"B00L9EPT8O,B01E6AO69U",Amazon 9W PowerFast Official OEM USB Charger a...,Amazon
9,"B00L9EPT8O,B01E6AO69U","Amazon Fire Hd 10 Tablet, Wi-Fi, 16 Gb, Specia...",Amazon
10,"B00L9EPT8O,B01E6AO69U",Amazon Fire Hd 6 Standing Protective Case(4th ...,Amazon
13,"B00L9EPT8O,B01E6AO69U",Amazon Fire Tv,Amazon
14,"B00L9EPT8O,B01E6AO69U",Amazon Kindle Fire 5ft USB to Micro-USB Cable ...,Amazon
24,"B00L9EPT8O,B01E6AO69U",Coconut Water Red Tea 16.5 Oz (pack of 12),Amazon
25,"B00L9EPT8O,B01E6AO69U",Echo (Black),Amazon
32,"B00L9EPT8O,B01E6AO69U","Kindle Dx Leather Cover, Black (fits 9.7 Displ...",Amazon
39,"B00L9EPT8O,B01E6AO69U",New Amazon Kindle Fire Hd 9w Powerfast Adapter...,Amazon



Cluster 2
Top words: kindle, voyage, reading, tablets, readers, kindle voyage, light, read, great, amazon, easy, page, love, books, screen


...

,product_id,product_name,brand
2,B01J2G4VBG,"All-New Kindle E-reader - Black, 6 Glare-Free ...",Amazon
6,B01AHB9C1E,Amazon 5W USB Official OEM Charger and Power A...,Amazon
15,B01J2G4VBG,Amazon Kindle Fire Hd (3rd Generation) 8gb,Amazon



Cluster 4
Top words: tablet, great, tablets, use, bought, books, love, loves, price, apps, amazon, kindle, screen, easy, good


,product_id,product_name,brand
0,B01AHB9CN2,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon
1,B01AHB9CYG,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon
5,B018Y229OU,Amazon - Kindle Voyage - 4GB - Wi-Fi + 3G - Black,Amazon
17,B018Y23MNM,Amazon Kindle Paperwhite - eBook reader - 4 GB...,Amazon
20,B018Y229OU,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...,Amazon
26,B018SZT3BK,Echo (White),Amazon
28,B01AHB9CYG,"Fire HD 8 Tablet with Alexa, 8 HD Display, 32 ...",Amazon
29,B018Y225IA,"Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16...",Amazon



Cluster 5
Top words: cover, case, kindle, light, kindle keyboard, lighted, keyboard, leather, new, covers, new case, kindle cover, lighted cover, kindle 5th, 5th generation


,product_id,product_name,brand
16,B005PB2T0S,Amazon Kindle Lighted Leather Cover,Amazon
33,B005PB2T0S,Kindle Keyboard,Amazon


In [27]:
product_df["cluster_text_v2"] = (
    product_df["product_name"].fillna("") + " " +
    product_df["category"].fillna("")
)

tfidf_v2 = TfidfVectorizer(
    stop_words="english",
    max_features=3000,
    ngram_range=(1, 2)
)

X_products_v2 = tfidf_v2.fit_transform(
    product_df["cluster_text_v2"]
)

for k in [4, 5, 6]:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = model.fit_predict(X_products_v2)

    score = silhouette_score(
        X_products_v2,
        labels
    )

    print(f"k={k}, silhouette score={score:.4f}")

k=4, silhouette score=0.2815


k=5, silhouette score=0.2781


k=6, silhouette score=0.2904


In [28]:
kmeans_v2 = KMeans(
    n_clusters=6,
    random_state=42,
    n_init=20
)

product_df["cluster"] = kmeans_v2.fit_predict(X_products_v2)

print(product_df["cluster"].value_counts().sort_index())

cluster
0     3
1    12
2    10
3     7
4     3
5     5
Name: count, dtype: int64


In [29]:
for cluster in range(6):
    print(f"\n===== CLUSTER {cluster} =====")

    display(
        product_df.loc[
            product_df["cluster"] == cluster,
            ["product_id", "product_name", "brand", "category"]
        ]
    )


===== CLUSTER 0 =====


,product_id,product_name,brand,category
0,B01AHB9CN2,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta..."
1,B01AHB9CYG,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Tablets,Fire Tablets,Electronics,Computers,Com..."
28,B01AHB9CYG,"Fire HD 8 Tablet with Alexa, 8 HD Display, 32 ...",Amazon,"Tablets,Fire Tablets,Electronics,Computers,Com..."



===== CLUSTER 1 =====


,product_id,product_name,brand,category
3,"B00L9EPT8O,B01E6AO69U",Amazon - Amazon Tap Portable Bluetooth and Wi-...,Amazon,"Stereos,Remote Controls,Amazon Echo,Audio Dock..."
6,B01AHB9C1E,Amazon 5W USB Official OEM Charger and Power A...,Amazon,"Tablets,Fire Tablets,Computers & Tablets,All T..."
7,"B00L9EPT8O,B01E6AO69U",Amazon 9W PowerFast Official OEM USB Charger a...,Amazon,"Stereos,Remote Controls,Amazon Echo,Audio Dock..."
9,"B00L9EPT8O,B01E6AO69U","Amazon Fire Hd 10 Tablet, Wi-Fi, 16 Gb, Specia...",Amazon,"Stereos,Remote Controls,Amazon Echo,Audio Dock..."
10,"B00L9EPT8O,B01E6AO69U",Amazon Fire Hd 6 Standing Protective Case(4th ...,Amazon,"Stereos,Remote Controls,Amazon Echo,Audio Dock..."
13,"B00L9EPT8O,B01E6AO69U",Amazon Fire Tv,Amazon,"Stereos,Remote Controls,Amazon Echo,Audio Dock..."
14,"B00L9EPT8O,B01E6AO69U",Amazon Kindle Fire 5ft USB to Micro-USB Cable ...,Amazon,"Stereos,Remote Controls,Amazon Echo,Audio Dock..."
24,"B00L9EPT8O,B01E6AO69U",Coconut Water Red Tea 16.5 Oz (pack of 12),Amazon,"Stereos,Remote Controls,Amazon Echo,Audio Dock..."
25,"B00L9EPT8O,B01E6AO69U",Echo (Black),Amazon,"Stereos,Remote Controls,Amazon Echo,Audio Dock..."
26,B018SZT3BK,Echo (White),Amazon,"Fire Tablets,Tablets,Computers & Tablets,All T..."



===== CLUSTER 2 =====


...

,product_id,product_name,brand,category
15,B01J2G4VBG,Amazon Kindle Fire Hd (3rd Generation) 8gb,Amazon,"Amazon Devices & Accessories,Amazon Device Acc..."
17,B018Y23MNM,Amazon Kindle Paperwhite - eBook reader - 4 GB...,Amazon,"Tablets,Fire Tablets,Computers & Tablets,All T..."
18,B018Y225IA,Amazon Kindle Touch Leather Case (4th Generati...,Amazon,"Computers/Tablets & Networking,Tablets & eBook..."
20,B018Y229OU,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...,Amazon,"Fire Tablets,Tablets,Computers & Tablets,All T..."
29,B018Y225IA,"Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16...",Amazon,"Computers/Tablets & Networking,Tablets & eBook..."
30,B00IOY8XWQ,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",Amazon,"Walmart for Business,Office Electronics,Tablet..."
31,B00ZV9PXP2,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",Amazon,"Electronics,iPad & Tablets,All Tablets,Compute..."



===== CLUSTER 4 =====


,product_id,product_name,brand,category
2,B01J2G4VBG,"All-New Kindle E-reader - Black, 6 Glare-Free ...",Amazon,"Amazon Devices & Accessories,Amazon Device Acc..."
16,B005PB2T0S,Amazon Kindle Lighted Leather Cover,Amazon,"Electronics,eBook Readers & Accessories,Covers..."
33,B005PB2T0S,Kindle Keyboard,Amazon,"Electronics,eBook Readers & Accessories,Covers..."



===== CLUSTER 5 =====


,product_id,product_name,brand,category
5,B018Y229OU,Amazon - Kindle Voyage - 4GB - Wi-Fi + 3G - Black,Amazon,"Fire Tablets,Tablets,Computers & Tablets,All T..."
19,B018Y229OU,Amazon Standing Protective Case for Fire HD 6 ...,Amazon,"Fire Tablets,Tablets,Computers & Tablets,All T..."
21,B018Y229OU,Certified Refurbished Amazon Fire TV (Previous...,Amazon,"Fire Tablets,Tablets,Computers & Tablets,All T..."
27,B018Y229OU,"Fire HD 8 Tablet with Alexa, 8 HD Display, 16 ...",Amazon,"Fire Tablets,Tablets,Computers & Tablets,All T..."
34,B018Y229OU,Kindle Oasis E-reader with Leather Charging Co...,Amazon,"Fire Tablets,Tablets,Computers & Tablets,All T..."


In [30]:
terms = tfidf_v2.get_feature_names_out()

centers = kmeans_v2.cluster_centers_.argsort()[:, ::-1]

for cluster in range(6):
    top_words = [
        terms[i]
        for i in centers[cluster, :15]
    ]

    print(f"Cluster {cluster}:")
    print(", ".join(top_words))
    print()

Cluster 0:
tablets, hd, tablets tablets, hd display, hd tablet, computers, new hd, tablet hd, offers magenta, magenta, gb, tablet, gb includes, components hard, hard

Cluster 1:
home, smart, audio, smart home, speakers, amazon, echo, automation, home automation, hubs, security, smart hubs, clearance, assistants, wireless

Cluster 2:
readers, tablets, voyage, kindle voyage, kindle, wi, fi, wi fi, electronics, amazon kindle, amazon, office electronics, walmart business, walmart, business

Cluster 3:
tablets, readers, computers, computers tablets, tablets tablets, accessories, kindle, ebook, electronics, tablets computers, ebook readers, tablets ebook, readers computers, tablets networking, networking tablets

Cluster 4:
kindle, accessories, kindle 5th, 5th generation, 5th, covers, accessories kindle, kindle reader, generation, reader, reader accessories, electronics ebook, store amazon, accessories covers, generation covers

Cluster 5:
tablets, tablets tablets, tablets android, music ele

In [31]:
kmeans_4 = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=20
)

product_df["cluster_4"] = kmeans_4.fit_predict(X_products_v2)

print(
    product_df["cluster_4"]
    .value_counts()
    .sort_index()
)

cluster_4
0     3
1     9
2    16
3    12
Name: count, dtype: int64


In [32]:
for cluster in range(4):
    print(f"\n===== CLUSTER {cluster} =====")

    cluster_products = product_df.loc[
        product_df["cluster_4"] == cluster,
        ["product_id", "product_name", "brand"]
    ]

    display(cluster_products)


===== CLUSTER 0 =====


,product_id,product_name,brand
2,B01J2G4VBG,"All-New Kindle E-reader - Black, 6 Glare-Free ...",Amazon
16,B005PB2T0S,Amazon Kindle Lighted Leather Cover,Amazon
33,B005PB2T0S,Kindle Keyboard,Amazon



===== CLUSTER 1 =====


,product_id,product_name,brand
4,B00IOY8XWQ,"Amazon - Fire 16GB (5th Gen, 2015 Release) - B...",Amazon
8,B00IOY8XWQ,Amazon Echo and Fire TV Power Adapter,Amazon
11,B00IOY8XWQ,Amazon Fire Hd 8 8in Tablet 16gb Black B018szt...,Amazon
12,B00IOY8XWQ,"Amazon Fire Kids Edition Tablet, 7 Display, Wi...",Amazon
22,B00IOY8XWQ,Certified Refurbished Amazon Fire TV Stick (Pr...,Amazon
23,B00IOY8XWQ,Certified Refurbished Amazon Fire TV with Alex...,Amazon
36,B00IOY8XWQ,Kindle Paperwhite,Amazon
37,B00IOY8XWQ,"Kindle Paperwhite E-reader - White, 6 High-Res...",Amazon
38,B00OQVZDJM,"Kindle Voyage E-reader, 6 High-Resolution Disp...",Amazon



===== CLUSTER 2 =====


,product_id,product_name,brand
0,B01AHB9CN2,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon
1,B01AHB9CYG,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon
5,B018Y229OU,Amazon - Kindle Voyage - 4GB - Wi-Fi + 3G - Black,Amazon
15,B01J2G4VBG,Amazon Kindle Fire Hd (3rd Generation) 8gb,Amazon
17,B018Y23MNM,Amazon Kindle Paperwhite - eBook reader - 4 GB...,Amazon
18,B018Y225IA,Amazon Kindle Touch Leather Case (4th Generati...,Amazon
19,B018Y229OU,Amazon Standing Protective Case for Fire HD 6 ...,Amazon
20,B018Y229OU,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...,Amazon
21,B018Y229OU,Certified Refurbished Amazon Fire TV (Previous...,Amazon
27,B018Y229OU,"Fire HD 8 Tablet with Alexa, 8 HD Display, 16 ...",Amazon



===== CLUSTER 3 =====


,product_id,product_name,brand
3,"B00L9EPT8O,B01E6AO69U",Amazon - Amazon Tap Portable Bluetooth and Wi-...,Amazon
6,B01AHB9C1E,Amazon 5W USB Official OEM Charger and Power A...,Amazon
7,"B00L9EPT8O,B01E6AO69U",Amazon 9W PowerFast Official OEM USB Charger a...,Amazon
9,"B00L9EPT8O,B01E6AO69U","Amazon Fire Hd 10 Tablet, Wi-Fi, 16 Gb, Specia...",Amazon
10,"B00L9EPT8O,B01E6AO69U",Amazon Fire Hd 6 Standing Protective Case(4th ...,Amazon
13,"B00L9EPT8O,B01E6AO69U",Amazon Fire Tv,Amazon
14,"B00L9EPT8O,B01E6AO69U",Amazon Kindle Fire 5ft USB to Micro-USB Cable ...,Amazon
24,"B00L9EPT8O,B01E6AO69U",Coconut Water Red Tea 16.5 Oz (pack of 12),Amazon
25,"B00L9EPT8O,B01E6AO69U",Echo (Black),Amazon
26,B018SZT3BK,Echo (White),Amazon


In [33]:
terms = tfidf_v2.get_feature_names_out()

centers_4 = kmeans_4.cluster_centers_.argsort()[:, ::-1]

for cluster in range(4):
    top_words = [
        terms[i]
        for i in centers_4[cluster, :15]
    ]

    print(f"Cluster {cluster}:")
    print(", ".join(top_words))
    print()

Cluster 0:
kindle, accessories, kindle 5th, 5th generation, 5th, covers, accessories kindle, kindle reader, generation, reader, reader accessories, electronics ebook, store amazon, accessories covers, generation covers

Cluster 1:
tablets, readers, kindle voyage, voyage, kindle, wi fi, fi, wi, electronics, amazon kindle, amazon, office electronics, business, business office, electronics tablets

Cluster 2:
tablets, tablets tablets, computers, readers, computers tablets, electronics, tablets computers, hd, kindle, ipad, ipad tablets, electronics ipad, toys movies, movies music, music

Cluster 3:
home, smart, audio, smart home, speakers, amazon, echo, automation, home automation, hubs, security, smart hubs, clearance, assistants, wireless



In [34]:
name_text = (
    product_df["product_name"]
    .replace("Unknown", "")
    .fillna("")
)

product_df["cluster_text_v3"] = (
    (name_text + " ") * 4
    + product_df["category"].fillna("")
)

In [35]:
tfidf_v3 = TfidfVectorizer(
    stop_words="english",
    max_features=3000,
    ngram_range=(1, 2)
)

X_products_v3 = tfidf_v3.fit_transform(
    product_df["cluster_text_v3"]
)

In [36]:
for k in [4, 5, 6]:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = model.fit_predict(X_products_v3)

    score = silhouette_score(
        X_products_v3,
        labels
    )

    print(f"k={k}, silhouette score={score:.4f}")

k=4, silhouette score=0.1026


k=5, silhouette score=0.1231


k=6, silhouette score=0.1385


In [37]:
kmeans_v3 = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=20
)

product_df["cluster_v3"] = kmeans_v3.fit_predict(
    X_products_v3
)

for cluster in range(4):
    print(f"\n===== CLUSTER {cluster} =====")

    display(
        product_df.loc[
            product_df["cluster_v3"] == cluster,
            ["product_id", "product_name"]
        ]
    )


===== CLUSTER 0 =====


,product_id,product_name
3,"B00L9EPT8O,B01E6AO69U",Amazon - Amazon Tap Portable Bluetooth and Wi-...
6,B01AHB9C1E,Amazon 5W USB Official OEM Charger and Power A...
7,"B00L9EPT8O,B01E6AO69U",Amazon 9W PowerFast Official OEM USB Charger a...
9,"B00L9EPT8O,B01E6AO69U","Amazon Fire Hd 10 Tablet, Wi-Fi, 16 Gb, Specia..."
10,"B00L9EPT8O,B01E6AO69U",Amazon Fire Hd 6 Standing Protective Case(4th ...
13,"B00L9EPT8O,B01E6AO69U",Amazon Fire Tv
14,"B00L9EPT8O,B01E6AO69U",Amazon Kindle Fire 5ft USB to Micro-USB Cable ...
24,"B00L9EPT8O,B01E6AO69U",Coconut Water Red Tea 16.5 Oz (pack of 12)
25,"B00L9EPT8O,B01E6AO69U",Echo (Black)
26,B018SZT3BK,Echo (White)



===== CLUSTER 1 =====


,product_id,product_name
4,B00IOY8XWQ,"Amazon - Fire 16GB (5th Gen, 2015 Release) - B..."
5,B018Y229OU,Amazon - Kindle Voyage - 4GB - Wi-Fi + 3G - Black
8,B00IOY8XWQ,Amazon Echo and Fire TV Power Adapter
11,B00IOY8XWQ,Amazon Fire Hd 8 8in Tablet 16gb Black B018szt...
12,B00IOY8XWQ,"Amazon Fire Kids Edition Tablet, 7 Display, Wi..."
15,B01J2G4VBG,Amazon Kindle Fire Hd (3rd Generation) 8gb
16,B005PB2T0S,Amazon Kindle Lighted Leather Cover
17,B018Y23MNM,Amazon Kindle Paperwhite - eBook reader - 4 GB...
18,B018Y225IA,Amazon Kindle Touch Leather Case (4th Generati...
20,B018Y229OU,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...



===== CLUSTER 2 =====


,product_id,product_name
0,B01AHB9CN2,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,..."
1,B01AHB9CYG,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,..."
19,B018Y229OU,Amazon Standing Protective Case for Fire HD 6 ...
27,B018Y229OU,"Fire HD 8 Tablet with Alexa, 8 HD Display, 16 ..."
28,B01AHB9CYG,"Fire HD 8 Tablet with Alexa, 8 HD Display, 32 ..."



===== CLUSTER 3 =====


,product_id,product_name
2,B01J2G4VBG,"All-New Kindle E-reader - Black, 6 Glare-Free ..."
31,B00ZV9PXP2,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes..."
34,B018Y229OU,Kindle Oasis E-reader with Leather Charging Co...
35,B00VINDBJK,Kindle Oasis E-reader with Leather Charging Co...
37,B00IOY8XWQ,"Kindle Paperwhite E-reader - White, 6 High-Res..."
38,B00OQVZDJM,"Kindle Voyage E-reader, 6 High-Resolution Disp..."


In [38]:
kmeans_6_v3 = KMeans(
    n_clusters=6,
    random_state=42,
    n_init=20
)

product_df["cluster_v3_6"] = kmeans_6_v3.fit_predict(
    X_products_v3
)

print(
    product_df["cluster_v3_6"]
    .value_counts()
    .sort_index()
)

cluster_v3_6
0     4
1     3
2     4
3     3
4    12
5    14
Name: count, dtype: int64


In [39]:
for cluster in range(6):
    print(f"\n===== CLUSTER {cluster} =====")

    display(
        product_df.loc[
            product_df["cluster_v3_6"] == cluster,
            ["product_id", "product_name"]
        ]
    )


===== CLUSTER 0 =====


,product_id,product_name
34,B018Y229OU,Kindle Oasis E-reader with Leather Charging Co...
35,B00VINDBJK,Kindle Oasis E-reader with Leather Charging Co...
37,B00IOY8XWQ,"Kindle Paperwhite E-reader - White, 6 High-Res..."
38,B00OQVZDJM,"Kindle Voyage E-reader, 6 High-Resolution Disp..."



===== CLUSTER 1 =====


,product_id,product_name
21,B018Y229OU,Certified Refurbished Amazon Fire TV (Previous...
22,B00IOY8XWQ,Certified Refurbished Amazon Fire TV Stick (Pr...
23,B00IOY8XWQ,Certified Refurbished Amazon Fire TV with Alex...



===== CLUSTER 2 =====


...

,product_id,product_name
16,B005PB2T0S,Amazon Kindle Lighted Leather Cover
18,B018Y225IA,Amazon Kindle Touch Leather Case (4th Generati...
33,B005PB2T0S,Kindle Keyboard



===== CLUSTER 4 =====


,product_id,product_name
3,"B00L9EPT8O,B01E6AO69U",Amazon - Amazon Tap Portable Bluetooth and Wi-...
6,B01AHB9C1E,Amazon 5W USB Official OEM Charger and Power A...
7,"B00L9EPT8O,B01E6AO69U",Amazon 9W PowerFast Official OEM USB Charger a...
9,"B00L9EPT8O,B01E6AO69U","Amazon Fire Hd 10 Tablet, Wi-Fi, 16 Gb, Specia..."
10,"B00L9EPT8O,B01E6AO69U",Amazon Fire Hd 6 Standing Protective Case(4th ...
13,"B00L9EPT8O,B01E6AO69U",Amazon Fire Tv
14,"B00L9EPT8O,B01E6AO69U",Amazon Kindle Fire 5ft USB to Micro-USB Cable ...
24,"B00L9EPT8O,B01E6AO69U",Coconut Water Red Tea 16.5 Oz (pack of 12)
25,"B00L9EPT8O,B01E6AO69U",Echo (Black)
26,B018SZT3BK,Echo (White)



===== CLUSTER 5 =====


,product_id,product_name
2,B01J2G4VBG,"All-New Kindle E-reader - Black, 6 Glare-Free ..."
4,B00IOY8XWQ,"Amazon - Fire 16GB (5th Gen, 2015 Release) - B..."
5,B018Y229OU,Amazon - Kindle Voyage - 4GB - Wi-Fi + 3G - Black
8,B00IOY8XWQ,Amazon Echo and Fire TV Power Adapter
11,B00IOY8XWQ,Amazon Fire Hd 8 8in Tablet 16gb Black B018szt...
12,B00IOY8XWQ,"Amazon Fire Kids Edition Tablet, 7 Display, Wi..."
15,B01J2G4VBG,Amazon Kindle Fire Hd (3rd Generation) 8gb
17,B018Y23MNM,Amazon Kindle Paperwhite - eBook reader - 4 GB...
19,B018Y229OU,Amazon Standing Protective Case for Fire HD 6 ...
20,B018Y229OU,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...


In [40]:
terms_v3 = tfidf_v3.get_feature_names_out()

centers_v3 = kmeans_6_v3.cluster_centers_.argsort()[:, ::-1]

for cluster in range(6):
    top_words = [
        terms_v3[i]
        for i in centers_v3[cluster, :15]
    ]

    print(f"Cluster {cluster}:")
    print(", ".join(top_words))
    print()

Cluster 0:
300 ppi, resolution display, resolution, 300, high resolution, high, display 300, ppi, fi includes, includes special, includes, offers kindle, reader, special, special offers

Cluster 1:
refurbished amazon, refurbished, certified, certified refurbished, amazon tv, tv, previous, generation 1st, 1st, previous generation, 1st certified, amazon, tablets, generation, tv previous

Cluster 2:
hd, hd display, hd tablet, gb, tablet, offers, special offers, special, display, tablets, gb tangerine, tangerine special, tangerine, alexa hd, tablet alexa

Cluster 3:
kindle, kindle keyboard, keyboard, leather, generation, kindle 5th, 5th generation, lighted, kindle lighted, lighted leather, keyboard kindle, covers, 5th, leather cover, accessories kindle

Cluster 4:
home, smart, audio, amazon, smart home, speakers, echo, usb, home automation, security, smart hubs, hubs, automation, clearance, assistants

Cluster 5:
tablets, kindle, readers, amazon, wi, fi, wi fi, black, amazon kindle, electr

---

### CHANGE 3 — a way to tell which variant is actually better

Silhouette measures how tight and separated the clusters are. It never checks whether they
are the **right** groups, and on this data it points the wrong way: it rates
`name + category` highly, and that variant turns out to be no better than random.

So below: a correct answer written by hand, **ARI** to score against it, and sentence
embeddings as a fifth option.

In [41]:
### CHANGE 3a — write the correct answer by hand, so variants can be measured.
#
# WHY: without something to compare against, "these clusters look sensible" is
# the only available judgement. These five categories were assigned by reading
# the 39 product names, before any clustering was run.
from sklearn.metrics import adjusted_rand_score

READER = ("paperwhite", "voyage", "oasis", "e-reader", "ereader",
          "ebook", "keyboard")


def gold_label(n):
    n = str(n).lower()
    if "coconut" in n:
        return "EXCLUDE"
    if any(w in n for w in ("charger", "adapter", "cable")):
        return "Accessories"
    if ("cover" in n or "case" in n) and not (
            "tablet" in n or any(w in n for w in READER)):
        return "Accessories"
    if "fire tv" in n:
        return "Streaming"
    if "echo" in n or "tap" in n:
        return "Smart speakers"
    if any(w in n for w in READER):
        return "E-readers"
    if "kindle" in n and "fire" not in n:
        return "E-readers"
    return "Tablets"


product_df["gold"] = product_df["product_name"].apply(gold_label)
product_df = product_df[product_df["gold"] != "EXCLUDE"].reset_index(drop=True)
gold = product_df["gold"].values

print(product_df["gold"].value_counts().to_string())
print("\n%d products labelled by hand" % len(product_df))

gold
Tablets           13
Accessories       10
E-readers          9
Streaming          4
Smart speakers     3

39 products labelled by hand


In [42]:
### CHANGE 3b — embed the product name with a sentence model.
#
# WHY: TF-IDF can only measure SHARED WORDS. "Echo (White)" and "Amazon Tap
# Portable Bluetooth and Wi-Fi Speaker" are both speakers and share no word at
# all, so no TF-IDF variant - including the name x4 trick - can ever group them.
# A sentence model places text by meaning instead.
#
# 33M parameters, about a second for 39 products, and transformers is already
# installed. Four models were compared and this one won; the 110M model did not.
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel

torch.set_num_threads(2)
EMB_MODEL = "sentence-transformers/all-MiniLM-L12-v2"

tok = AutoTokenizer.from_pretrained(EMB_MODEL)
mod = AutoModel.from_pretrained(EMB_MODEL).eval()

names = product_df["product_name"].fillna("").tolist()
enc = tok(names, padding=True, truncation=True, max_length=64,
          return_tensors="pt")
with torch.no_grad():
    hidden = mod(**enc).last_hidden_state

mask = enc["attention_mask"].unsqueeze(-1).float()
X_emb = ((hidden * mask).sum(1) / mask.sum(1)).numpy()
X_emb = X_emb / np.linalg.norm(X_emb, axis=1, keepdims=True)

print("embedded %d names -> %d dimensions" % X_emb.shape)

C:\Users\Windows\python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  23%|██▎       | 45/199 [00:00<00:00, 414.32it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1196.54it/s]

embedded 39 names -> 384 dimensions


In [43]:
### CHANGE 3c — score every variant the same way, by ARI.
#
# WHY: this is the comparison the notebook was missing. Silhouette is printed
# alongside only to show that it disagrees - and that it is wrong.
from sklearn.metrics import silhouette_score

tf_v1 = TfidfVectorizer(stop_words="english", max_features=3000,
                        ngram_range=(1, 2)).fit_transform(
                            product_df["cluster_text"])
tf_v3 = TfidfVectorizer(stop_words="english", max_features=3000,
                        ngram_range=(1, 2)).fit_transform(
                            (product_df["product_name"].fillna("") + " ") * 4
                            + product_df["category"].fillna(""))
tf_name = TfidfVectorizer(stop_words="english", max_features=3000,
                          ngram_range=(1, 2)).fit_transform(
                              product_df["product_name"].fillna(""))

OPTIONS = {
    "v1  name+brand+category+reviews": tf_v1,
    "v3  name x4 + category":          tf_v3,
    "    name only":                   tf_name,
    "    MiniLM embeddings":           X_emb,
}

print("%-34s %3s %12s %8s" % ("representation", "k", "silhouette", "ARI"))
print("=" * 62)
best = (None, None, -1)
for tag, X in OPTIONS.items():
    for k in (4, 5, 6):
        lab = KMeans(n_clusters=k, random_state=42, n_init=20).fit_predict(X)
        ari = adjusted_rand_score(gold, lab)
        print("%-34s %3d %12.4f %8.4f"
              % (tag if k == 4 else "", k, silhouette_score(X, lab), ari))
        if ari > best[2]:
            best = (tag, k, ari)
    print("-" * 62)

print()
print("BEST: %s at k=%d, ARI %.4f" % best)
print("Silhouette would have picked a different - and worse - option.")

BEST_X = OPTIONS[best[0]]
BEST_K = best[1]
product_df["cluster_best"] = KMeans(n_clusters=BEST_K, random_state=42,
                                    n_init=20).fit_predict(BEST_X)

representation                       k   silhouette      ARI


v1  name+brand+category+reviews      4       0.1316   0.0894


                                     5       0.1258   0.1091


                                     6       0.1414   0.1133
--------------------------------------------------------------


v3  name x4 + category               4       0.1130   0.2264
                                     5       0.1248   0.2978


                                     6       0.1379   0.2195
--------------------------------------------------------------
    name only                        4       0.0734   0.2680
                                     5       0.0813   0.3790


                                     6       0.0875   0.5409
--------------------------------------------------------------
    MiniLM embeddings                4       0.1740   0.5642
                                     5       0.1982   0.6380


                                     6       0.2158   0.7811
--------------------------------------------------------------

BEST:     MiniLM embeddings at k=6, ARI 0.7811
Silhouette would have picked a different - and worse - option.


In [44]:
### CHANGE 4 — derive the names from each cluster's contents.
#
# WHY: KMeans numbers its clusters arbitrarily, and the numbering changes
# whenever the input does - a different k, one product more, another seed. A
# dictionary keyed on 0..5 then mislabels everything and says nothing about it.
# This exact bug hit our own notebook: the cluster whose biggest product was
# "Echo (White)" came out labelled "Chargers & cables".
#
# Naming from the dominant real category instead means the labels follow the
# data, and the assertion fails loudly if the clustering ever shifts.
POWER = ("charger", "adapter", "cable", "powerfast", "usb")

cluster_names = {}
for i in range(BEST_K):
    members = product_df[product_df["cluster_best"] == i]
    cluster_names[i] = members["gold"].value_counts().index[0]

# accessories split into two groups at this k, which is more useful than one
for i, nm in list(cluster_names.items()):
    if nm != "Accessories":
        continue
    txt = " ".join(product_df.loc[product_df["cluster_best"] == i,
                                  "product_name"].astype(str)).lower()
    power = sum(txt.count(w) for w in POWER)
    cases = txt.count("case") + txt.count("cover")
    cluster_names[i] = "Chargers & cables" if power > cases else "Cases & covers"

product_df["meta_category"] = product_df["cluster_best"].map(cluster_names)

assert len(set(cluster_names.values())) == BEST_K, "names must be unique"

print("%-7s %-26s %9s  %s" % ("cluster", "name", "products", "largest member"))
print("-" * 92)
for i in range(BEST_K):
    m = product_df[product_df["cluster_best"] == i]
    pure = 100 * m["gold"].value_counts().iloc[0] / len(m)
    print("%-7d %-26s %9d  %-34s %.0f%% pure"
          % (i, cluster_names[i], len(m),
             str(m["product_name"].iloc[0])[:34], pure))

display(
    product_df[
        ["product_id", "product_name", "cluster_best", "meta_category", "gold"]
    ].sort_values("meta_category")
)

cluster name                        products  largest member
--------------------------------------------------------------------------------------------
0       Tablets                           13  All-New Fire HD 8 Tablet, 8 HD Dis 100% pure
1       E-readers                          8  All-New Kindle E-reader - Black, 6 100% pure
2       Streaming                          5  Amazon Echo and Fire TV Power Adap 80% pure
3       Smart speakers                     2  Echo (Black)                       100% pure
4       Cases & covers                     7  Amazon - Amazon Tap Portable Bluet 71% pure
5       Chargers & cables                  4  Amazon 5W USB Official OEM Charger 100% pure


,product_id,product_name,cluster_best,meta_category,gold
3,"B00L9EPT8O,B01E6AO69U",Amazon - Amazon Tap Portable Bluetooth and Wi-...,4,Cases & covers,Smart speakers
10,"B00L9EPT8O,B01E6AO69U",Amazon Fire Hd 6 Standing Protective Case(4th ...,4,Cases & covers,Accessories
16,B005PB2T0S,Amazon Kindle Lighted Leather Cover,4,Cases & covers,Accessories
18,B018Y225IA,Amazon Kindle Touch Leather Case (4th Generati...,4,Cases & covers,Accessories
19,B018Y229OU,Amazon Standing Protective Case for Fire HD 6 ...,4,Cases & covers,Accessories
31,"B00L9EPT8O,B01E6AO69U","Kindle Dx Leather Cover, Black (fits 9.7 Displ...",4,Cases & covers,Accessories
32,B005PB2T0S,Kindle Keyboard,4,Cases & covers,E-readers
6,B01AHB9C1E,Amazon 5W USB Official OEM Charger and Power A...,5,Chargers & cables,Accessories
7,"B00L9EPT8O,B01E6AO69U",Amazon 9W PowerFast Official OEM USB Charger a...,5,Chargers & cables,Accessories
14,"B00L9EPT8O,B01E6AO69U",Amazon Kindle Fire 5ft USB to Micro-USB Cable ...,5,Chargers & cables,Accessories


---
## What to check in the output

**How many products are named `Unknown`.** His saved run has **11 of 38**, because he
groups by `product_id` and many ids carry no name. Nine of those eleven land in
`Accessories & Power`, which makes that cluster largely a bucket of unidentified products
rather than a category.

**Whether the names are still corrupted.** Watch for `Echo (White),,,\r\nEcho (White),,,`
— he did not apply the name repair, so the duplication goes straight into the clustering
text.

**How many distinct products he actually has.** 38 `product_id` values collapse to **15
distinct names**: six separate ids all called "Fire Tablet, 7 Display… Black", five
"Kindle Fire 16gb… Blue", four "Echo (White)". Our Step 0 established that `id`/`asins`
is not a reliable product key, and this is what that looks like downstream.

### The one-line improvement

In his `product_df = df_work.groupby("product_id")`, group by **`product_name`** instead —
using the repaired names from our Step 0. That alone should empty the `Unknown` bucket and
take the product count from 38 ambiguous ids to 40 real products.